# 01 Download MedRAG and EU health PDFs

Raw layer: Hugging Face MedRAG JSON plus health PDF files on disk.


In [ ]:
import runpy
import sys
from pathlib import Path
runpy.run_path(str((Path.cwd() / "_bootstrap.py") if (Path.cwd() / "_bootstrap.py").exists() else (Path.cwd() / "notebooks" / "_bootstrap.py")))
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
for pth in (PROJECT_ROOT, PROJECT_ROOT / "app"):
    if str(pth) not in sys.path:
        sys.path.insert(0, str(pth))
from scripts.load_dataset import create_medical_seed
from ingestion.paths import MEDRAG_RAW_DIR, ensure_data_dirs
ensure_data_dirs()
create_medical_seed(
    dataset_path="MedRAG/textbooks",
    seed_size=500,
    output_path=str(MEDRAG_RAW_DIR / "medical_textbook_seed.json"),
    output_format="json",
    source="textbook",
)
create_medical_seed(
    dataset_path="MedRAG/pubmed",
    seed_size=500,
    output_path=str(MEDRAG_RAW_DIR / "medical_pubmed_seed.json"),
    output_format="json",
    source="pubmed",
)
print("MedRAG done", MEDRAG_RAW_DIR)


In [ ]:
import json
import requests
from ingestion.paths import PDF_RAW_DIR, ensure_data_dirs
EU_PDF_SOURCES = [
 {"filename": "eu_ctr_536_2014_guide.pdf", "url": "https://health.ec.europa.eu/document/download/f5ad2a13-4a41-4ada-81a1-2854783c75c0_en?filename=mp_ctr-536-2014_guide_en.pdf", "category": "clinical"},
 {"filename": "eu_ctr_536_2014_qa.pdf", "url": "https://health.ec.europa.eu/document/download/bd165522-8acf-433a-9ab1-d7dceae58112_en?filename=regulation5362014_qa_en.pdf", "category": "clinical"},
 {"filename": "eu_regulation_536_2014_eurlex.pdf", "url": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:02014R0536-20221205", "category": "clinical"},
 {"filename": "eu_amr_surveillance_2023.pdf", "url": "https://www.ecdc.europa.eu/sites/default/files/documents/antimicrobial-consumption-ESAC-Net-annual-epidemiological-report-2023_0.pdf", "category": "diseases"},
]
HEADERS = {"User-Agent": "MedJurisRAG/1.0", "Accept": "application/pdf,*/*"}

def download_one(entry, out_dir, timeout=120):
    dest = out_dir / entry["filename"]
    if dest.exists() and dest.stat().st_size > 10000:
        print("exists", entry["filename"]); return dest
    try:
        r = requests.get(entry["url"], headers=HEADERS, timeout=timeout, stream=True)
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(65536):
                if chunk: f.write(chunk)
        if dest.stat().st_size < 5000:
            dest.unlink(missing_ok=True); print("small", entry["filename"]); return None
        print("ok", entry["filename"]); return dest
    except Exception as e:
        print("fail", entry["filename"], e); dest.unlink(missing_ok=True); return None

ensure_data_dirs(); PDF_RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest, downloaded = [], []
for entry in EU_PDF_SOURCES:
    path = download_one(entry, PDF_RAW_DIR)
    manifest.append({**entry, "ok": path is not None})
    if path: downloaded.append(path)
(PDF_RAW_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(len(downloaded), "/", len(EU_PDF_SOURCES), "PDFs in", PDF_RAW_DIR)

